# Equação de Poisson 3D em Eletrostática — Resolução Método Galerkin-Colocação

## 1. A Equação de Poisson em Eletrostática

### 1.1 Formulação física

Em eletrostática, o potencial elétrico $\phi(\mathbf{x})$ gerado por uma distribuição de carga $\rho(\mathbf{x})$ satisfaz a equação de Poisson:

$$
\boxed{\nabla^2 \phi(\mathbf{x}) = -\frac{\rho(\mathbf{x})}{\varepsilon_0}}
$$

onde:
- $\phi(\mathbf{x})$ é o **potencial elétrico** (em Volts)
- $\rho(\mathbf{x})$ é a **densidade volumétrica de carga** (em C/m³)
- $\varepsilon_0 \approx 8.854 \times 10^{-12}$ F/m é a **permissividade do vácuo**

### 1.2 Interpretação física

- **Linearidade**: o potencial é linear na densidade de carga, pelo princípio da superposição.
- **Ausência de singularidade no interior**: no interior do domínio, $\phi$ é suave se $\rho$ for suave. Singularidades só aparecem quando $\rho$ é uma delta de Dirac (carga puntual).
- **Domínio fixo**: consideramos $\Omega = [a_x, b_x] \times [a_y, b_y] \times [a_z, b_z]$, com $a_x = a_y = a_z = -1$ e $b_x = b_y = b_z = 1$ neste desenvolvimento.
- **Condições de contorno**: nas faces do domínio, o potencial satisfaz as condições de contorno de Dirichlet, por exemplo, as faces são condutores aterrados ($\phi = 0$) ou mantidos em um potencial fixo.

### 1.3 Significado da fonte

Seja $\nabla^2 u = f$ com

$$
f(\mathbf{x}) = -\frac{\rho(\mathbf{x})}{\varepsilon_0}.
$$

A escolha que reproduz o caso do notebook corresponde a uma densidade de carga proporcional a uma autofunção do Laplaciano no cubo $[-1,1]^3$, que se anula em todas as faces.

## 2. Dedução Analítica por Separação de Variáveis

### 2.1 A equação

Resolvemos

$$
\nabla^2 \phi(x,y,z) = -\frac{\rho(x,y,z)}{\varepsilon_0}, \qquad (x,y,z) \in \Omega = [-1,1]^3,
$$

com condições de contorno de Dirichlet homogêneas:

$$
\phi = 0 \quad \text{em } \partial\Omega \ \text{(faces aterradas)}.
$$

### 2.2 Autofunções do Laplaciano no cubo $[-1,1]^3$

As autofunções do operador $-\nabla^2$ no cubo $[-1,1]^3$ com condições de Dirichlet homogêneas são:

$$
\boxed{
\psi_{nml}(x,y,z) = \sin\!\left(\frac{n\pi(x+1)}{2}\right)
\sin\!\left(\frac{m\pi(y+1)}{2}\right)
\sin\!\left(\frac{l\pi(z+1)}{2}\right),
\qquad n,m,l \in \mathbb{N}^*
}
$$

com autovalores

$$
\boxed{
\Lambda_{nml} = \frac{\pi^2}{4}(n^2 + m^2 + l^2)
}
$$

Essas funções formam uma base completa de $L^2(\Omega)$, ou seja, qualquer função suave que se anula nas faces pode ser expandida nessa base. Note que a normalização por $(x+1)/2$ mapeia $[-1,1]$ para $[0,1]$, e a autofunção se anula em $x = \pm 1$, $y = \pm 1$, $z = \pm 1$.

### 2.3 Expansão do potencial e da carga

Expandimos o potencial e a densidade de carga:

$$
\phi = \sum_{n,m,l} \Phi_{nml}\,\psi_{nml},
\qquad
\rho = \sum_{n,m,l} R_{nml}\,\psi_{nml},
$$

onde os coeficientes da carga são as projeções

$$
R_{nml} = \int_{-1}^{1}\!\!\int_{-1}^{1}\!\!\int_{-1}^{1}
\rho(x,y,z)\,\psi_{nml}(x,y,z)\,dx\,dy\,dz
\;\Big/\; \|\psi_{nml}\|^2,
$$

sendo $\|\psi_{nml}\|^2 = 1$ para a base ortonormalizada (ou $1$ para a base não normalizada, conforme a convenção).

Substituindo em $\nabla^2 \phi = -\rho/\varepsilon_0$:

$$
\sum_{n,m,l} \Phi_{nml}\,(-\Lambda_{nml})\,\psi_{nml}
= -\frac{1}{\varepsilon_0}\sum_{n,m,l} R_{nml}\,\psi_{nml}.
$$

Pela independência linear das autofunções, igualamos coeficiente a coeficiente:

$$
\boxed{
\Phi_{nml} = \frac{R_{nml}}{\varepsilon_0\,\Lambda_{nml}}
= \frac{4\,R_{nml}}{\varepsilon_0\,\pi^2(n^2+m^2+l^2)}
}
$$

O Laplaciano age diagonalmente na base de autofunções e o potencial é obtido dividindo o coeficiente de carga pelo autovalor correspondente.

### 2.4 Solução geral em série

$$
\boxed{
\phi(x,y,z) = \frac{1}{\varepsilon_0}\sum_{n,m,l=1}^{\infty}
\frac{4\,R_{nml}}{\pi^2(n^2+m^2+l^2)}
\sin\!\left(\frac{n\pi(x+1)}{2}\right)
\sin\!\left(\frac{m\pi(y+1)}{2}\right)
\sin\!\left(\frac{l\pi(z+1)}{2}\right)
}
$$

Que é o análogo 3D da série de Fourier senoidal para o problema eletrostático de Dirichlet no cubo $[-1,1]^3$.

## 3. Exemplo a ser Implementado

### 3.1 Densidade de carga

No código, escolhemos uma **gaussiana deslocada** no domínio $[-1,1]^3$:

$$
\rho(x,y,z) = A \exp\!\left(-\frac{(x-x_0)^2 + (y-y_0)^2 + (z-z_0)^2}{2\sigma^2}\right),
$$

com centro $(x_0, y_0, z_0) = (0.3, -0.2, 0.4)$, largura $\sigma = 0.15$ e amplitude $A = 1{,}0 \times 10^{-10}$ C/m³. Essa função é **suave e analítica** no interior do domínio, mas **não é autofunção do Laplaciano** no cubo com condições de Dirichlet homogêneas — portanto, **não admite solução analítica fechada** por separação de variáveis. A integral de convolução com a função de Green do cubo não possui forma elementar, e a solução só pode ser obtida numericamente.

A escolha do centro fora da origem e a largura moderada fazem com que a solução não herde a simetria do cubo — o pico de $\phi$ fica deslocado em direção ao ponto $(0.3, -0.2, 0.4)$, e as superfícies equipotenciais deixam de ser simétricas em relação aos planos $x=0$, $y=0$, $z=0$.

### 3.2 Potencial resultante

Como a fonte não é autofunção, o potencial é obtido como solução numérica do sistema linear

$$
\mathcal{L}\,\phi = f, \qquad f(\mathbf{x}) = -\frac{\rho(\mathbf{x})}{\varepsilon_0},
$$

com as linhas de fronteira substituídas por $\phi_i = 0$ (Dirichlet homogêneo). O resultado não tem forma fechada, mas herda as propriedades qualitativas esperadas:

- **Máximo positivo** próximo ao centro da gaussiana, em $(0.3, -0.2, 0.4)$, com valor que decai conforme a distância à fonte aumenta.
- **Decaimento suave** em direção às seis faces, onde $\phi \to 0$.
- **Assimetria** em relação aos planos coordenados, reflexo direto do deslocamento da fonte.
- **Ausência de oscilações espúrias** para $N$ suficientemente grande, graças à suavidade de $\rho$.

A validação contra solução analítica deixa de ser possível diretamente, mas o **resíduo da equação de Poisson nos pontos interiores** continua sendo uma métrica confiável: para $N \gtrsim 20$, o resíduo deve ficar próximo do erro de máquina ($\sim 10^{-14}$), confirmando que a solução numérica é precisa.

### 3.3 Interpretação física

- **Fonte localizada**: a gaussiana deslocada representa uma nuvem de carga positiva concentrada em torno de um ponto no interior do cubo, mas **fora do centro geométrico**. Fisicamente, é o análogo de uma impureza carregada ou de uma distribuição de carga confinada em uma região específica do domínio.
- **Quebra de simetria**: como o centro não coincide com a origem, o potencial **não é simétrico** sob reflexão $x \to -x$, $y \to -y$ ou $z \to -z$. Isso é o oposto do caso senoidal, onde a simetria do cubo era preservada.
- **Sinal**: a carga é positiva em todo o domínio (a gaussiana nunca troca de sinal), então o potencial também é positivo, com máximo próximo ao centro da fonte e decaimento para zero nas faces.
- **Amplitude**: a amplitude do potencial é proporcional a $A/\varepsilon_0$, mas seu valor exato depende da geometria da fonte e das condições de contorno — sendo **necessariamente obtido por via numérica**, já que não há fórmula fechada disponível.
- **Relevância prática**: este é o caso mais próximo de uma aplicação real, onde a distribuição de carga não tem simetria especial e o método espectral numérico é **essencial** — a separação de variáveis que funcionou para a senoide não se aplica mais.

### 3.4 Por que essa escolha é adequada

- **Força o uso do método numérico**: elimina a possibilidade de "resolver à mão" e valida o pipeline como ferramenta de solução.
- **Mantém a suavidade necessária**: a gaussiana é $C^\infty$ e analítica, então a **convergência espectral permanece exponencial** — não há degradação para algébrica como no caso de fontes com descontinuidade.
- **Não muda a base de Chebyshev**: a base continua sendo $\{T_0, \dots, T_N\}$ em cada direção, com produto tensorial. O que muda é **quantos polinômios** são necessários para atingir precisão de máquina — para $\sigma = 0.15$, $N \approx 20$ já deve ser suficiente; para $\sigma = 0.02$ (camada limite), seria necessário $N \gtrsim 60$.
- **Prepara o pipeline para fontes arbitrárias**: uma vez validado com a gaussiana deslocada, o pipeline está pronto para aceitar qualquer densidade suave definida no `parameters.yml` — incluindo combinações de gaussianas, polinômios, ou fontes tabuladas.

## 4. Cuidados ao Aplicar o Método Pseudo-Espectral

- **Mapeamento de domínio**:
Chebyshev é definido em $[-1,1]$; aplicamos em cada direção
$$
x_j = \frac{a_x+b_x}{2} + \frac{b_x-a_x}{2}\xi_j, \quad
y_k = \frac{a_y+b_y}{2} + \frac{b_y-a_y}{2}\xi_k, \quad
z_\ell = \frac{a_z+b_z}{2} + \frac{b_z-a_z}{2}\xi_\ell.
$$
Com $a=-1$ e $b=1$, o mapeamento reduz-se à identidade em cada direção, e os pontos de colocação coincidem com os pontos GLC de Chebyshev no domínio físico.

- **Matriz de diferenciação**:
Usamos $D_x^{(2)} = D_x \cdot D_x$ (produto matricial), e analogamente para $y$ e $z$, para preservar a precisão em $N$ moderado.

- **Operador Laplaciano discreto**:
O Laplaciano 3D em malha tensorial é construído via **soma de Kronecker**:
$$
\mathcal{L} = D_x^{(2)} \otimes I_y \otimes I_z + I_x \otimes D_y^{(2)} \otimes I_z + I_x \otimes I_y \otimes D_z^{(2)}.
$$
Isso evita construir manualmente matrizes 3D densas e preserva a estrutura tensorial.

- **Imposição das condições de contorno**:
Substituímos as linhas correspondentes aos pontos de fronteira do sistema $\mathcal{L}\phi = f$ pelas condições de Dirichlet ($\phi_i = 0$, faces aterradas em $x=\pm 1$, $y=\pm 1$, $z=\pm 1$). Isso destrói a estrutura espectral nessas linhas, mas é adequado para problemas suaves.

- **Pontos de colocação GLC**:
Em cada direção, há densidade maior nas bordas, o que é vantajoso para capturar gradientes próximos às faces do paralelepípedo.

- **Suavidade da fonte $\rho$**:
A convergência espectral exige $\rho$ analítica. A gaussiana deslocada satisfaz essa condição e garante convergência exponencial do erro. Descontinuidades na densidade de carga degradam a convergência de exponencial para algébrica e podem gerar oscilações espúrias.

- **Custo computacional**:
A matriz $\mathcal{L}$ tem tamanho $(N_x+1)(N_y+1)(N_z+1) \times \cdots$, ou seja, cresce rápido com $N$. Para $N=20$ em cada direção, são $\sim 9000$ incógnitas; para $N=40$, $\sim 69000$. O custo de LU denso é $\mathcal{O}(M^3)$ com $M$ o número total de incógnitas, o que mostra um rápido crescimento das operações a serem processadas.

## 5. Implicações para a Estrutura da Solução

### 5.1 Paridade nas faces (caso de domínio simétrico)

Se o domínio for simétrico em relação a um plano, por exemplo $x=0$, e $\rho$ for **par** em $x$, então:

- $\phi(\mathbf{x})$ é **par** em $x$: $\phi(x,y,z) = \phi(-x,y,z)$
- $\partial_x \phi$ é **ímpar** em $x$: $\partial_x \phi(0,y,z) = 0$
- A condição de regularidade em $x=0$ é naturalmente satisfeita por uma base com apenas potências pares em $x$

Nesse caso, usamos **apenas polinômios de Chebyshev de ordem par** $T_{2k}$ na direção $x$, o que melhora a convergência perto do plano de simetria.

**No caso da gaussiana deslocada**, essa simetria **não está presente** (o centro não coincide com nenhum plano de simetria do cubo), então usamos **toda a base** $T_0, \dots, T_N$ em cada direção. A ausência de paridade não é um problema — apenas significa que não podemos explorar essa redução para diminuir o custo.

### 5.2 Caso geral em $\Omega = [a_x,b_x]\times[a_y,b_y]\times[a_z,b_z]$

Como não há restrição de paridade, usamos toda a base $T_0, \dots, T_{N_x}$ em cada direção, e analogamente para $y$ e $z$.

### 5.3 Condições de contorno

Para condições de contorno de Dirichlet homogêneas ($\phi=0$ em $\partial\Omega$), as linhas da matriz correspondentes aos pontos de fronteira são substituídas por $\phi_i = 0$. Para condições de contorno de Dirichlet não-homogêneas, $\phi_i = V(\mathbf{x}_i)$ diferente de zero, o potencial pode ser prescrito nas faces (condutores mantidos a potenciais fixos).

## 6. Formulação Espectral

No código, resolvemos o mesmo problema da dedução analítica, mas em vez de expandir em autofunções analíticas (que só existem para geometrias simples e fontes específicas), expandimos em polinômios de Chebyshev e impomos as condições de contorno por substituição direta dos valores nas linhas da matriz.

| Elemento | Código espectral |
|---|---|
| Base de expansão | $T_k(x)T_l(y)T_m(z)$ |
| Operador| $\mathcal{L}$ (Kronecker de $D^{(2)}$) |
| Coeficientes do potencial| $\phi = \mathcal{L}^{-1} b$ |
| Condições de Contorno| impostas por substituição direta |
| Aplicabilidade| geometrias e fontes arbitrárias |

Para a **gaussiana deslocada** não existe expansão fechada em autofunções com apenas alguns modos. É exatamente por isso que o método espectral numérico se torna **indispensável**: ele resolve o mesmo tipo de problema sem depender da existência de uma base analítica conveniente. A **validação cruzada** com o caso senoidal (que tem gabarito) garante que o solver está correto; a partir daí, ele pode ser aplicado com confiança a fontes arbitrárias como a gaussiana.

## 7. Tabela Resumo

| Aspecto | Equação de Poisson 3D em Eletrostática |
|---|---|
| Equação | $\nabla^2 \phi = -\rho/\varepsilon_0$ |
| Natureza | Linear, desacoplada |
| Singularidade no interior | Nenhuma para $\rho$ suave |
| Domínio | Fixo $\Omega = [-1,1]^3$ |
| Condições de contorno | Dirichlet ($\phi=0$, faces aterradas) |
| Densidade de carga | Gaussiana deslocada: $A\exp\!\left(-\frac{|\mathbf{x}-\mathbf{c}|^2}{2\sigma^2}\right)$ |
| Solução analítica | **Não disponível** — fonte não é autofunção |
| Autofunções (para referência) | $\psi_{nml} = \sin\!\left(\frac{n\pi(x+1)}{2}\right)\sin\!\left(\frac{m\pi(y+1)}{2}\right)\sin\!\left(\frac{l\pi(z+1)}{2}\right)$ |
| Autovalores (para referência) | $\Lambda_{nml} = \frac{\pi^2}{4}(n^2+m^2+l^2)$ |
| Polinômios de Chebyshev | Expansão em cada direção, base completa $T_0, \dots, T_N$ |
| Convergência esperada | Exponencial (fonte analítica) |
| $N$ necessário (estimado) | $\sim 20$ para $\sigma = 0.15$ |
| Métrica de validação | Resíduo interior $\|\mathcal{L}\phi - f\|_\infty$ |

## 8. Conclusão

A equação de Poisson 3D em eletrostática, com uma densidade de carga gaussiana deslocada, é um caso onde o método espectral é **realmente necessário**:

1. **Sem singularidades:** a gaussiana é suave e analítica, o que mantém a convergência exponencial do erro.
2. **Domínio fixo:** sem autovalor de fronteira livre.
3. **Linear:** resolvemos com uma única fatoração LU.
4. **Estrutura tensorial:** o Laplaciano é construído por soma de Kronecker.
5. **Sem paridade explorável:** o deslocamento da fonte quebra todas as simetrias do cubo, então usamos a base completa $T_0, \dots, T_N$.
6. **Ausência de gabarito analítico:** ao contrário do caso senoidal, não há fórmula fechada para $\phi$ — a validação é feita pelo **resíduo da equação de Poisson**, que deve cair para $\sim 10^{-14}$ com $N$ suficientemente grande.

O pipeline mantém a mesma estrutura: monta o Laplaciano tensorial, impõe as condições de contorno de Dirichlet por substituição de linha, resolve o sistema linear denso e valida o resultado pelo resíduo interior. A diferença em relação ao caso senoidal é que o gabarito analítico é substituído pela **consistência numérica** (resíduo) e pela **comparação com o caso senoidal previamente validado** — estabelecendo a confiança necessária para usar o pipeline como gerador de dados de treinamento do MLP em situações onde a solução analítica é desconhecida.